# 5. Проверка устойчивости спецификаций (многоканальный случай)

**Цель:** сопоставить несколько допустимых спецификаций `GraphEVTPipeline` на Kuramoto-ряде с впрыснутым событием, включая выбор `persistence` и метода графа, и сверить не только обнаружение, но и оценённый источник с известной истиной.

In [1]:
from pathlib import Path
from importlib.metadata import version
import json
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/kuramoto_propagation_series.csv").exists())
DATA = ROOT / "data/generated/kuramoto_propagation_series.csv"
SEED = 42
df = pd.read_csv(DATA)
channel_names = [column for column in df.columns if column.startswith("channel_")]
x = df[channel_names].to_numpy()
event_mask = df["is_extreme"].to_numpy(dtype=bool)
assert np.isfinite(x).all() and x.ndim == 2

propagation_metadata = json.loads((ROOT / "data/generated/synthetic_datasets_metadata.json").read_text())["kuramoto_propagation"]
event_onset = int(np.flatnonzero(event_mask)[0])
true_source_name = channel_names[int(propagation_metadata["source_channel"])]

In [2]:
BASELINE_SIZE = event_onset
specifications = [
    {"role": "persistence_1_ar", "tail_quantile": 0.85, "alarm_probability": 0.97, "persistence": 1, "graph_method": "ar"},
    {"role": "persistence_2_ar", "tail_quantile": 0.85, "alarm_probability": 0.97, "persistence": 2, "graph_method": "ar"},
    {"role": "conservative_persistence_2", "tail_quantile": 0.90, "alarm_probability": 0.99, "persistence": 2, "graph_method": "ar"},
    {"role": "persistence_1_dfa", "tail_quantile": 0.85, "alarm_probability": 0.97, "persistence": 1, "graph_method": "dfa"},
]
rows = []
for spec in specifications:
    evt = EVTConfig(
        baseline_size=BASELINE_SIZE,
        tail_quantile=spec["tail_quantile"],
        alarm_probability=spec["alarm_probability"],
        persistence=spec["persistence"],
        top_k=3,
    )
    run = GraphEVTPipeline(
        evt,
        graph=GraphConfig(method=spec["graph_method"], edge_threshold=0.35, random_state=SEED),
        input_config=InputConfig(missing="error", require_regular_time=True),
        verbose=True,
    ).run_detailed(x)
    estimated_source = None if run.ranking is None else channel_names[run.ranking.source]
    rows.append({
        **spec,
        "alarm_threshold": run.detection.alarm_threshold,
        "post_baseline_exceedances": int(np.sum(run.detection.indicator[BASELINE_SIZE:] > run.detection.alarm_threshold)),
        "detected": run.detection.detected,
        "first_persistent_index": run.detection.time_index,
        "delay_samples": None if run.detection.time_index is None else run.detection.time_index - event_onset,
        "graph_available": run.graph is not None,
        "estimated_source": estimated_source,
        "source_correct": estimated_source == true_source_name,
    })
sensitivity = pd.DataFrame(rows)
sensitivity

Graph-EVT pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

Graph-EVT pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

Graph-EVT pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

Graph-EVT pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

,role,tail_quantile,alarm_probability,persistence,graph_method,alarm_threshold,post_baseline_exceedances,detected,first_persistent_index,delay_samples,graph_available,estimated_source,source_correct
0,persistence_1_ar,0.85,0.97,1,ar,1.076386,57,True,949,17,True,channel_00,True
1,persistence_2_ar,0.85,0.97,2,ar,1.076386,57,True,959,27,True,channel_01,False
2,conservative_persistence_2,0.90,0.99,2,ar,1.082979,44,True,959,27,True,channel_01,False
3,persistence_1_dfa,0.85,0.97,1,dfa,1.076386,57,True,949,17,True,channel_00,True


In [3]:
artifact_dir = ROOT / "results/n-d"
artifact_dir.mkdir(parents=True, exist_ok=True)
sensitivity.to_csv(artifact_dir / "05_specification_review.csv", index=False)

agreement = {
    "all_detected": bool(sensitivity["detected"].all()),
    "none_detected": bool((~sensitivity["detected"]).all()),
    "same_first_index": bool(sensitivity["first_persistent_index"].nunique(dropna=False) == 1),
    "all_source_correct": bool(sensitivity["source_correct"].all()),
    "fraction_source_correct": float(sensitivity["source_correct"].mean()),
    "library_version": version("graph-evt-agent"),
}
agreement

{'all_detected': True,
 'none_detected': False,
 'same_first_index': False,
 'all_source_correct': False,
 'fraction_source_correct': 0.5,
 'library_version': '0.1.0'}

Все четыре спецификации согласны в обнаружении, но **не в локализации**: `persistence=1` (со стандартным и с DFA-графом) находит устойчивое превышение достаточно рано, пока сигнал источника ещё доминирует, и корректно называет истинный канал; `persistence=2` требует двух подряд превышений, срабатывание сдвигается позже (импульс уже затухает и распространился на соседей), и оценённым источником становится ближайший сосед по кольцу, а не сам источник. Это не брак библиотеки — это ожидаемая чувствительность эвристики ranking к моменту, в который она получает управление.

**Самопроверка:** почему более строгая спецификация (`persistence=2`, `tail_quantile=0.90`) не является "более правильной" по умолчанию? Что произошло бы с точностью локализации при ещё большей задержке обнаружения? Почему согласие по одному ряду всё ещё не образует benchmark?